# 线程与进程

学习目标：按任务的等待、计算和数据共享需求选择线程或进程，组织并发工作，并正确处理结果、失败与资源退出。

前置知识：函数与参数、对象引用、异常处理、with、模块导入、程序入口、pickle 和 subprocess。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

进程示例在独立脚本中启动，正常完成后等待并关闭工作进程。

配套脚本：位于 [scripts/26-threads-and-processes/](scripts/26-threads-and-processes/)。

（1）[process\_demo.py](scripts/26-threads-and-processes/process_demo.py)：观察 spawn、列表副本、进程队列与退出清理。

（2）[pool\_demo.py](scripts/26-threads-and-processes/pool_demo.py)：观察进程池、可序列化任务与 Future 异常回传。

## 1 并发与并行

并发（concurrency）关注多个任务在一段时间内共同推进；并行（parallelism）关注任务是否在同一时刻执行。启动多个工作单元，并不自动证明获得了并行加速。

CPython 3.12 的全局解释器锁（GIL）使同一进程内通常只有一个线程执行 Python 代码；等待 I/O 的任务仍可通过线程重叠等待。多个进程可以利用多个 CPU 核心，但启动和传递数据也有成本。这里先保留一个很小的顺序计算，后面只核对结果，不比较耗时。

| 工具 | 中文名称／含义 | 本章用途 |
| --- | --- | --- |
| threading.Thread | 线程 | 共享进程内对象，显式启动与等待 |
| queue.Queue | 线程安全队列 | 在线程之间移交工作 |
| multiprocessing.Process | 进程 | 在独立进程中执行函数 |
| ThreadPoolExecutor | 线程池执行器 | 通过 Future 取得任务结果 |
| ProcessPoolExecutor | 进程池执行器 | 执行可序列化的计算任务 |

本章关注多个执行单元同时访问这些对象时的协调。

In [1]:
def square(number: int) -> int:
    """计算一个整数的平方。"""
    return number * number


numbers = [2, 3, 4]
expected_squares = [square(number) for number in numbers]
print(expected_squares)  # [4, 9, 16]：后续线程池应得到相同结果。

[4, 9, 16]


## 2 线程的启动、等待与失败

### 2.1 target 传函数，start 启动线程

Thread 的 target 接收可调用对象，args 接收位置参数元组；写成 target=函数调用会先在当前线程调用它。start 每个 Thread 只能调用一次，join 等待线程退出，不返回 target 的返回值。

下面把结果写入显式传入的列表，主线程在 join 后读取；这里只有一个写入者。线程使用非守护模式，不能靠解释器退出时突然停止工作来代替清理。

In [2]:
import threading


def record_minutes(minutes: int, records: list[int]) -> None:
    """把一条学习时长写入调用方提供的列表。"""
    records.append(minutes)


records = []
worker = threading.Thread(target=record_minutes, args=(25, records))
worker.start()
worker.join(timeout=3)
assert not worker.is_alive()
print(records)  # [25]：线程看见的是同一个列表。


[25]


In [3]:
worker.start()  # RuntimeError：同一个 Thread 不能再次启动。

RuntimeError: threads can only be started once

### 2.2 join 超时不会停止线程

join 无论正常完成还是超时都返回 None；需要继续用 is\_alive 判断。Event 用一个标记在线程间通知：set 发出通知，wait 等待标记并返回是否等到。

示例先让工作线程等待通知，再用 join(timeout=0) 立即检查。工作函数的等待也设置有限超时；finally 发出退出通知并再次 join，避免遗留线程。timeout 的单位是秒。

In [4]:
def wait_for_release(
    started: threading.Event,
    release: threading.Event,
) -> str:
    """通知已经开始，收到放行信号后结束，超时则明确失败。"""
    # 用两个事件区分“已经开始”和“可以结束”，不靠 sleep 猜测执行时机。
    started.set()
    if not release.wait(timeout=3):
        raise TimeoutError("工作线程未在 3 秒内收到放行信号")
    return "完成"


started = threading.Event()
release = threading.Event()
waiting_worker = threading.Thread(
    target=wait_for_release, args=(started, release),
)
waiting_worker.start()
try:
    assert started.wait(timeout=3)
    print(waiting_worker.join(timeout=0))  # None：返回值不表示完成。
    print(waiting_worker.is_alive())  # True：尚未放行。
finally:
    release.set()
    waiting_worker.join(timeout=4)
assert not waiting_worker.is_alive()

None
True


### 2.3 join 不会把目标函数异常重新抛给调用方

Thread.run 中未捕获的异常交给 threading.excepthook；默认会把通常的异常打印到标准错误流。join 只等待退出，不能用它判断业务成功。

下面临时替换 hook，保存完整格式化回溯，并在退出后恢复原 hook。保存字符串可避免长期保留异常对象及其回溯引用；需要像普通函数一样取得异常时，后面的 Future 更方便。

In [5]:
import functools
import traceback


def reject_record() -> None:
    """拒绝一条用于演示失败的学习记录。"""
    raise ValueError("学习分钟数不能为负数")


def save_thread_error(
    args: threading.ExceptHookArgs,
    reports: list[str],
) -> None:
    """保存线程异常的完整回溯文本。"""
    reports.append("".join(traceback.format_exception(
        args.exc_type, args.exc_value, args.exc_traceback,
    )))


error_reports = []
# 临时收集工作线程的回溯；join 只等待线程，不替它重新抛出业务异常。
original_hook = threading.excepthook
failed_worker = threading.Thread(target=reject_record)
threading.excepthook = functools.partial(
    save_thread_error, reports=error_reports,
)
try:
    failed_worker.start()
    failed_worker.join(timeout=3)  # 此处不会抛出目标函数的 ValueError。
finally:
    threading.excepthook = original_hook
assert not failed_worker.is_alive()
assert len(error_reports) == 1
assert error_reports[0].endswith("ValueError: 学习分钟数不能为负数\n")
print(error_reports[0], end="")  # 回溯应定位到 reject_record。

Traceback (most recent call last):
  File "C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\ZHUANG\AppData\Local\Temp\ipykernel_29032\777511256.py", line 7, in reject_record
    raise ValueError("学习分钟数不能为负数")
ValueError: 学习分钟数不能为负数


## 3 共享状态、竞态与锁

### 3.1 用屏障固定一次丢失更新

竞态条件（race condition）指结果取决于多个执行单元的交错顺序。读取旧值、计算新值、写回并不会因为有 GIL 就自动成为不可分割的操作；两个线程可能各自保存同一个旧值。

![丢失更新：两个线程写回同一个旧值](image/illustration/26-01-lost-update.svg)

图示：CMYK Labs 依据 threading 的屏障与共享状态机制自行绘制；由 Barrier 固定的丢失更新时序；图只展示一种写回先后，两者互换也得到相同结果。

Barrier 是固定数量线程的会合屏障，全部到达才能继续。下面用它保证两线程都读完旧值再写回，设置 3 秒等待超时，不靠随机休眠重现。观察结果后，对照锁版本检查完整的读、改、写是否位于锁内。

In [6]:
def increment_after_read(
    counter: dict[str, int],
    barrier: threading.Barrier,
) -> None:
    """先读取计数，再会合写回，用于展示丢失更新。"""
    previous = counter["value"]
    # 两条线程都读完旧值才放行，固定重现丢失更新。
    barrier.wait(timeout=3)
    counter["value"] = previous + 1


counter = {"value": 0}
barrier = threading.Barrier(2)
race_workers = [
    threading.Thread(target=increment_after_read, args=(counter, barrier))
    for _ in range(2)
]
for thread in race_workers:
    thread.start()
for thread in race_workers:
    thread.join(timeout=4)
assert all(not thread.is_alive() for thread in race_workers)
print(counter["value"])  # 1：两次写回都来自旧值 0，丢失了一次更新。
assert counter["value"] == 1

1


### 3.2 锁保护完整的读改写过程

Lock 同一时刻只允许一个持有者进入受保护区域；with 会在退出时释放锁。所有访问同一不变量的路径必须使用同一把锁，单独锁住赋值语句不足以保护前面的读取。

下面仍用屏障让工作线程先会合，再进入临界区（critical section），即受锁保护的代码段。不能把这个屏障放在锁内：第一个线程持锁等待第二个，第二个却无法进入。

In [7]:
import _thread


def increment_locked(
    counter: dict[str, int],
    lock: _thread.LockType,
    barrier: threading.Barrier,
) -> None:
    """先会合，再用同一把锁保护每次计数更新。"""
    # 先会合再加锁，避免一条线程持锁等待另一条无法进入的线程。
    barrier.wait(timeout=3)
    for _ in range(50):
        with lock:
            counter["value"] += 1


counter = {"value": 0}
lock = threading.Lock()
barrier = threading.Barrier(2)
locked_workers = [
    threading.Thread(target=increment_locked, args=(counter, lock, barrier))
    for _ in range(2)
]
for thread in locked_workers:
    thread.start()
for thread in locked_workers:
    thread.join(timeout=4)
assert all(not thread.is_alive() for thread in locked_workers)
print(counter["value"])  # 100：两条线程各完成 50 次更新。
assert counter["value"] == 100

100


## 4 用队列交接工作

### 4.1 生产者、消费者与结束哨兵

queue.Queue 提供线程安全的放入和取出操作。生产者提交数据，消费者循环取出并处理；本例约定 None 为结束哨兵（sentinel），因此 None 不能同时表示一项正常工作。有多个消费者时，每个都需要一份结束通知。

每次成功 get 后，在 finally 中调用一次 task\_done，包括哨兵。Queue.join 等待所有已放入项目被确认处理；Thread.join 等待消费者线程退出，两者等待的对象不同。确认处理结束也不代表业务一定成功，真实业务还需单独报告错误。

In [8]:
import queue


def consume_minutes(
    inbox: queue.Queue,
    totals: list[int],
) -> None:
    """累加队列里的学习时长，收到 None 后交回合计并退出。"""
    total = 0
    while True:
        minutes = inbox.get(timeout=3)
        try:
            if minutes is None:
                totals.append(total)
                return
            total += minutes
        # 每个 get 都必须配对 task_done，包括用于结束循环的 None。
        finally:
            inbox.task_done()


inbox = queue.Queue()
for minutes in [15, 20, 30, None]:
    inbox.put(minutes)  # 主线程是生产者；先提交有限输入与结束通知。
totals = []
consumer = threading.Thread(target=consume_minutes, args=(inbox, totals))
consumer.start()
consumer.join(timeout=4)
assert not consumer.is_alive()
assert totals == [65]  # 先确认消费者正常完成，再等待队列计数归零。
inbox.join()
print(totals)  # [65]：哨兵不计入总时长。

[65]


### 4.2 不用 empty 预测下一次 get

empty 和 qsize 只是观察当时状态，不能保证下一次操作仍然可行。需要立即尝试时，直接调用 get\_nowait 并处理 queue.Empty；有界队列放不下时对应 queue.Full。

task\_done 多调用会抛出 ValueError；少调用会让 join 一直等待。下面只运行能够立即结束的边界示例。

In [9]:
empty_inbox = queue.Queue()
empty_inbox.get_nowait()  # Empty：当前没有项目，直接展示原始异常。

Empty: 

In [10]:
empty_inbox.task_done()  # ValueError：没有对应的 put，计数不能减至负数。

ValueError: task_done() called too many times

## 5 避免死锁

### 5.1 普通 Lock 不可重入

死锁（deadlock）是执行单元彼此等待、无法继续推进的状态；同一线程重复等待自己持有的普通 Lock 也会卡住。RLock 允许同一线程重复获得锁，但每次获得仍需配对释放，也不能解决不同线程之间的循环等待。

下面用 acquire(blocking=False) 检查重复获得普通锁会失败，避免真的挂起。超时或非阻塞获取失败时，没有获得锁，不能为这次失败额外 release。

In [11]:
single_lock = threading.Lock()
with single_lock:
    acquired_again = single_lock.acquire(blocking=False)
    print(acquired_again)  # False：普通 Lock 不识别“是我自己”。
    assert not acquired_again
print(single_lock.locked())  # False：外层 with 已释放。

reentrant_lock = threading.RLock()
with reentrant_lock, reentrant_lock:
    print("同一线程可以再次进入")  # 两次进入各自配对释放。

False
False
同一线程可以再次进入


### 5.2 统一锁顺序，把等待结果移出工作线程

如果线程甲持有第一把锁并等待第二把，线程乙反过来等待，就可能形成循环。让所有路径按同一顺序获得锁，可以消除这种相反顺序造成的循环。

线程池也可能死锁：工作线程在只有一个工作槽的池内提交另一个任务，再等它的 result，后者便没有机会运行。把依赖安排在协调方；不要持锁做未知时长的 I/O，也不要用增加工作线程数代替检查依赖关系。下面只执行统一锁顺序的有限工作。

In [12]:
def record_with_two_locks(
    first: _thread.LockType,
    second: _thread.LockType,
    records: list[str],
    label: str,
) -> None:
    """按约定顺序获得两把锁后记录完成标记。"""
    with first, second:
        records.append(label)


first_lock = threading.Lock()
second_lock = threading.Lock()
finished_labels = []
# 所有线程传入相同的两把锁，并且获取顺序一致。
ordered_workers = [
    threading.Thread(
        target=record_with_two_locks,
        args=(first_lock, second_lock, finished_labels, label),
    )
    for label in ["A", "B"]
]
for thread in ordered_workers:
    thread.start()
for thread in ordered_workers:
    thread.join(timeout=3)
assert all(not thread.is_alive() for thread in ordered_workers)
print(sorted(finished_labels))  # ['A', 'B']：完成先后不作承诺。

['A', 'B']


## 6 线程池与 Future

### 6.1 提交任务并取得返回值

Executor.submit 返回 Future，代表一次调用的最终结果。ThreadPoolExecutor 复用有限数量的线程；with 退出时等待已提交工作并释放执行器资源。

下面沿用第一节的 square。按提交顺序读取 result 可以保持结果顺序，但不规定实际执行顺序；适合需要将每个结果对回原输入的场景。

In [13]:
import concurrent.futures

with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    square_futures = [executor.submit(square, number) for number in numbers]
    actual_squares = [future.result(timeout=3) for future in square_futures]
print(actual_squares)  # [4, 9, 16]
assert actual_squares == expected_squares

[4, 9, 16]


### 6.2 result 会重新抛出任务异常

Future.result 在任务成功时返回值，在任务失败时重新抛出该异常。不要只提交任务而从不检查结果，否则容易错过业务失败。

下面沿用 reject\_record，直接展示任务的原始 ValueError。线程池本身仍由 with 完成清理。

In [14]:
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    failed_future = executor.submit(reject_record)
    failed_future.result(timeout=3)  # ValueError：任务异常在读取结果时重新抛出。

ValueError: 学习分钟数不能为负数

In [15]:
print(failed_future.done())  # True：完成也可能表示失败；池已关闭。

True


### 6.3 wait 和 result 的超时只限制等待

wait 返回 done 和 not\_done 两个集合；超时后可能仍有未完成项，且不会自动取消它们。Future.result 超时则抛出 TimeoutError，同样不会停止底层调用。

下面仍用 Event 控制工作边界。timeout=0 使检查立即返回；finally 放行后，with 才能完成正常关闭。

In [16]:
started = threading.Event()
release = threading.Event()
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    held_future = executor.submit(wait_for_release, started, release)
    try:
        assert started.wait(timeout=3)
        done, not_done = concurrent.futures.wait([held_future], timeout=0)
        print(len(done), len(not_done))  # 0 1：任务还在等待通知。
        held_future.result(timeout=0)  # TimeoutError：只停止本次等待。
    finally:
        release.set()  # 原始异常继续传播；放行让 with 能等待并关闭池。

0 1


TimeoutError: 

In [17]:
print(held_future.result())  # 完成：超时没有停止底层调用。

完成


### 6.4 cancel 只能取消尚未开始的调用

Future.cancel 对已开始运行或已经完成的调用返回 False；尚未开始的调用可以取消。成功取消后，result 抛出 concurrent.futures.CancelledError。

示例只设置一个工作线程，先用事件确认第一个任务占据它，再提交第二个任务，因此取消边界由事件控制。Python 线程不能通过这个 API 被强制中断。

In [18]:
started = threading.Event()
release = threading.Event()
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    running_future = executor.submit(wait_for_release, started, release)
    try:
        assert started.wait(timeout=3)
        pending_future = executor.submit(square, 9)
        print(running_future.cancel(), pending_future.cancel())  # False True。
    finally:
        release.set()
assert running_future.result() == "完成"
assert pending_future.cancelled()

False True


In [19]:
pending_future.result()  # CancelledError：读取已取消调用的结果。

CancelledError: 

### 6.5 shutdown 的取消范围

shutdown(cancel\_futures=True) 取消尚未开始的 Future，保留正在运行的调用。wait=False 只让关闭方法先返回，资源会在任务完成后释放；它不意味着程序可以忽略仍在运行的任务。

这里显式调用 shutdown 观察这个边界，然后在 finally 放行并等待清理。关闭之后继续 submit 会抛出 RuntimeError；一般代码优先使用 with。

In [20]:
started = threading.Event()
release = threading.Event()
executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
running_future = executor.submit(wait_for_release, started, release)
try:
    assert started.wait(timeout=3)
    pending_future = executor.submit(square, 7)
    executor.shutdown(wait=False, cancel_futures=True)
    print(running_future.done(), pending_future.cancelled())  # False True。
finally:
    release.set()
    executor.shutdown(wait=True)
assert running_future.result() == "完成"

False True


In [21]:
executor.submit(square, 8)  # RuntimeError：已关闭的池不能再提交。

RuntimeError: cannot schedule new futures after shutdown

## 7 进程、序列化与 Windows 入口

### 7.1 显式选择 spawn 上下文

Python 3.12 的启动方式有平台差异。spawn 启动新的解释器，Windows 和 macOS 默认使用它；fork 复制父进程状态，仅适用于 POSIX，复制多线程进程存在风险；forkserver 由专门服务器派生进程，需要平台支持。

本章通过 get\_context("spawn") 显式选择方式，并从同一上下文创建 Process 和 Queue，不修改整个解释器的默认方式。子进程需要安全导入主模块，所以启动逻辑必须放在 if \_\_name\_\_ == "\_\_main\_\_": 保护的入口中；目标函数放在可导入模块顶层。

In [22]:
import multiprocessing

spawn_context = multiprocessing.get_context("spawn")
print(spawn_context.get_start_method())  # spawn：局部选择，不改全局默认。
assert "spawn" in multiprocessing.get_all_start_methods()
# 实际启动放在配套脚本；Notebook 的交互定义不作为 spawn 目标。

spawn

### 7.2 传递数据意味着序列化与重建

multiprocessing.Queue 用 pickle 序列化消息，接收端重建对象，不是直接共享原来的列表。spawn 的目标和参数也需要满足相应序列化要求；普通函数按模块名和限定名称定位，不能把 Notebook 局部函数或 lambda 当作可靠的进程任务入口。

先观察普通数据往返后的引用关系。这里只处理本程序构造的可信数据；不要反序列化未知来源的 pickle。为了减少跨进程传递成本，优先传需要的小型输入和结果。

In [23]:
import pickle

parent_record = {"minutes": [15, 20]}
received_record = pickle.loads(pickle.dumps(parent_record))
received_record["minutes"].append(30)
print(parent_record)  # {'minutes': [15, 20]}：原对象未被修改。
print(received_record)  # {'minutes': [15, 20, 30]}：修改发生在重建对象中。
assert received_record["minutes"] is not parent_record["minutes"]

{'minutes': [15, 20]}
{'minutes': [15, 20, 30]}


### 7.3 用 Process 和 Queue 完成一次数据往返

process\_demo.py 的顶层 append\_and\_send 修改子进程中的列表，再放入队列。main 先 get 接收消息，再 join 等待进程；反过来可能让父进程等待退出，而子进程等待队列缓冲被消费，形成死锁。

Process.join 后检查 exitcode；正常返回为 0，未捕获异常通常为 1。Process.close 释放已退出进程的句柄，不能关闭仍运行的进程。Queue.close 后调用 join\_thread，等待本进程的队列后台线程刷完缓冲。

脚本的 finally 在异常路径尝试终止仍存活的进程并等待退出；terminate 可能跳过子进程的 finally 并损坏队列，因此只作失败兜底，队列随后关闭且不复用。下面用当前解释器执行脚本，子进程沿用环境；外层命令超时为 40 秒，非零退出会明确报错。

```python
import json
import multiprocessing
import multiprocessing.queues


def append_and_send(
    minutes: list[int],
    output_queue: multiprocessing.queues.Queue,
) -> None:
    """修改子进程中的列表副本，并把结果交回父进程。"""
    try:
        minutes.append(30)
        output_queue.put(minutes)
    finally:
        output_queue.close()
        output_queue.join_thread()


def main() -> None:
    """启动子进程，先接收消息，再等待退出并释放句柄。"""
    # 1. 进程和队列来自同一个 spawn 上下文。
    context = multiprocessing.get_context("spawn")
    output_queue = context.Queue()
    parent_minutes = [15, 20]
    process = context.Process(
        target=append_and_send,
        args=(parent_minutes, output_queue),
    )
    process.start()
    try:
        # 2. 先取完消息，避免父进程等待退出、子进程等待管道被读取。
        child_minutes = output_queue.get(timeout=10)
        process.join(timeout=10)
        assert not process.is_alive()
        assert process.exitcode == 0
        # JSON 中 start_method=spawn、parent=[15, 20]、child=[15, 20, 30]、exitcode=0。
        print(json.dumps({
            "start_method": context.get_start_method(),
            "parent": parent_minutes,
            "child": child_minutes,
            "exitcode": process.exitcode,
        }))
    finally:
        # 3. 等待失败时终止仍存活的工作进程，再关闭本例不再使用的队列。
        if process.is_alive():
            process.terminate()
        process.join()
        process.close()
        output_queue.close()
        output_queue.join_thread()


if __name__ == "__main__":
    main()
```

Step 1：在课程目录运行独立脚本，输出应与下一单元注释一致。
```powershell
python scripts/26-threads-and-processes/process_demo.py
```

In [24]:
import json
import os
import subprocess
import sys
from pathlib import Path


def run_process_demo(filename: str) -> dict[str, object]:
    """运行本章的独立进程示例并读取 JSON 结果。"""
    script = Path("scripts/26-threads-and-processes") / filename
    child_environment = os.environ.copy()
    child_environment["PYTHONDONTWRITEBYTECODE"] = "1"
    # spawn 示例放在可导入脚本中，由独立进程启动，结果用 JSON 传回。
    completed = subprocess.run(
        [sys.executable, "-B", str(script)],
        capture_output=True,
        text=True,
        encoding="utf-8",
        timeout=40,
        check=True,
        env=child_environment,
    )
    return json.loads(completed.stdout)


process_report = run_process_demo("process_demo.py")
print(process_report)
# start_method 为 spawn，parent 为 [15, 20]，child 为 [15, 20, 30]。
assert process_report == {
    "start_method": "spawn",
    "parent": [15, 20],
    "child": [15, 20, 30],
    "exitcode": 0,
}

{'start_method': 'spawn', 'parent': [15, 20], 'child': [15, 20, 30], 'exitcode': 0}


### 7.4 用 ProcessPoolExecutor 汇总计算

进程池和线程池共享 submit、map、Future 接口，但进程池执行的函数、参数与返回值必须能够被 pickle 处理。主模块必须可由工作进程导入，所以本例仍从独立脚本启动。

pool\_demo.py 显式传入 spawn 上下文，计算从 0 到给定非负上界之前各整数的平方和，再检查负上界触发的 ValueError。map 按输入顺序产出结果；读取失败 Future 会在父进程重新抛出异常。不要在进程池工作函数中调用 Executor 或 Future 方法，这可能死锁。

进程池的 cancel 仍只针对未开始的 Future；任务可能已移交给工作进程，不要假设“尚未观察到输出”就一定可取消。with 等待全部有限任务并关闭池；这些小输入只展示接口和正确性，不支持性能结论。

```python
import concurrent.futures
import json
import multiprocessing


def sum_squares(limit: int) -> int:
    """返回从 0 到 limit 之前各整数的平方和，拒绝负上界。"""
    if limit < 0:
        raise ValueError("平方和上界不能为负数")
    return sum(number * number for number in range(limit))


def main() -> None:
    """执行小型计算，并在退出进程池前取回所有结果。"""
    context = multiprocessing.get_context("spawn")
    with concurrent.futures.ProcessPoolExecutor(
        max_workers=2,
        mp_context=context,
    ) as executor:
        # 1. map 的结果顺序与输入顺序一致，不依赖进程完成顺序。
        totals = list(executor.map(sum_squares, [3, 4, 5], timeout=15))
        failed = executor.submit(sum_squares, -1)
        # 2. 业务异常在父进程读取 Future 时重新抛出。
        try:
            failed.result(timeout=15)
        except ValueError as error:
            error_name = type(error).__name__
        else:
            raise AssertionError("负上界没有按预期失败")
    # with 已等待工作进程并关闭池；不将这个小例子当作跑分。
    # JSON 为 {"totals": [5, 14, 30], "error": "ValueError"}。
    print(json.dumps({"totals": totals, "error": error_name}))


if __name__ == "__main__":
    main()
```

Step 1：在课程目录运行独立脚本，输出应与下一单元注释一致。
```powershell
python scripts/26-threads-and-processes/pool_demo.py
```

In [25]:
pool_report = run_process_demo("pool_demo.py")
print(pool_report)  # {'totals': [5, 14, 30], 'error': 'ValueError'}
assert pool_report == {"totals": [5, 14, 30], "error": "ValueError"}
# 两个配套脚本均在自己的进程内创建和清理资源，不向内核留下工作进程。

{'totals': [5, 14, 30], 'error': 'ValueError'}


## 本章小结

（1）线程共享对象，适合重叠 I/O 等待；进程隔离普通对象，可以执行独立计算，但需要支付启动与数据传递成本。

（2）join 只负责等待退出。线程异常需要明确报告；Future.result 能返回值或重新抛出任务异常，done 不等于业务成功。

（3）锁保护完整不变量，队列移交工作。统一锁顺序、避免池内相互等待，并为每个任务安排结束通知和清理责任。

（4）超时只限制等待，取消只覆盖尚未开始的 Future。Windows 的 spawn 使用可导入顶层函数和入口保护，队列消息应先消费，再等待进程退出。

自查：能否分别说明“线程结束”“队列处理计数归零”“Future 成功”“进程句柄关闭”由哪一步确认？

## 练习

（1）先预测下面三个结果，再运行核对。说明线程共享引用与序列化重建之间的区别；检查标准是能逐项解释身份关系和列表内容。

In [26]:
exercise_original = [10, 20]
exercise_alias = exercise_original
exercise_copy = pickle.loads(pickle.dumps(exercise_original))
exercise_alias.append(30)
print(exercise_alias is exercise_original)
print(exercise_copy is exercise_original)
print(exercise_copy)
# 先写下预测，再核对每次赋值或序列化是否创建了新的列表。

True
False
[10, 20]


（2）沿用 increment\_locked，把线程数改为 3，每条线程仍更新 50 次，并使用匹配参与者数量的 Barrier。检查最终值为 150，所有线程退出，锁已经释放。

提示：屏障放在获得锁之前；不要运行一个参与者数量永远凑不齐、又没有超时的屏障。

In [27]:
exercise_counter = {"value": 0}
exercise_barrier = threading.Barrier(3)
exercise_lock = threading.Lock()
# 创建三条线程，显式传入同一组对象；启动后逐一 join 并核对边界。

（3）把学习时长 [10, 20, 30, 40] 分成两组，通过 ThreadPoolExecutor 提交 sum，并在协调方汇总。检查总数为 100，每个 Future 都已完成，退出 with 后不能再提交任务。

再说明若改为进程池，为什么把函数藏在 Notebook 内部并不足够；可参考 pool\_demo.py 的入口与顶层函数布局。

In [28]:
exercise_batches = [[10, 20], [30, 40]]
# 在 with 中提交两个 sum，读取结果并求总和。
# 在退出后用精确的 RuntimeError 反例检查提交边界，不遗留执行器。

### 练习提示与解析

（1）提示：赋值保留引用，pickle 往返重建列表。解析：依次为 True、False、[10, 20]；追加只改变原列表与它的别名。

（2）提示一：三个线程共享同一计数器、锁和三人屏障。提示二：先会合再持锁，主线程逐一 join。解析：三次独立的50次加一总计150；若屏障放入锁内，其余参与者不能持锁进入屏障，会等待到超时，不能以加大超时解决这种依赖冲突。

（3）提示：先收集两个 Future，再在协调方读取结果。解析：分别得到30和70，总数100；退出 with 后两个 Future 均 done，再 submit 直接产生 RuntimeError。进程池额外要求主模块可导入，并能序列化调用对象与数据，所以进程启动仍使用本章的脚本入口。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [threading 的 GIL 与 I/O 适用说明](https://docs.python.org/3.12/library/threading.html)；[Thread 的构造、启动与守护线程](https://docs.python.org/3.12/library/threading.html#thread-objects)、[join 超时](https://docs.python.org/3.12/library/threading.html#threading.Thread.join)、[线程异常 hook 与引用风险](https://docs.python.org/3.12/library/threading.html#threading.excepthook)、[Lock](https://docs.python.org/3.12/library/threading.html#lock-objects)、[锁类型标注](https://docs.python.org/3.12/library/_thread.html#thread.LockType)、[RLock](https://docs.python.org/3.12/library/threading.html#rlock-objects)、[Event](https://docs.python.org/3.12/library/threading.html#event-objects)、[Barrier 与超时](https://docs.python.org/3.12/library/threading.html#barrier-objects)；[Queue 与 Empty、Full](https://docs.python.org/3.12/library/queue.html#queue-objects)、[task\_done](https://docs.python.org/3.12/library/queue.html#queue.Queue.task_done)、[Queue.join](https://docs.python.org/3.12/library/queue.html#queue.Queue.join)；[Executor 的 submit、map 与 shutdown](https://docs.python.org/3.12/library/concurrent.futures.html#concurrent.futures.Executor)、[线程池及依赖死锁](https://docs.python.org/3.12/library/concurrent.futures.html#threadpoolexecutor)、[Future.result](https://docs.python.org/3.12/library/concurrent.futures.html#concurrent.futures.Future.result)、[Future.cancel](https://docs.python.org/3.12/library/concurrent.futures.html#concurrent.futures.Future.cancel)、[wait](https://docs.python.org/3.12/library/concurrent.futures.html#concurrent.futures.wait)、[进程池约束](https://docs.python.org/3.12/library/concurrent.futures.html#processpoolexecutor)；[启动方式与 get\_context](https://docs.python.org/3.12/library/multiprocessing.html#contexts-and-start-methods)、[Process 的退出码与清理](https://docs.python.org/3.12/library/multiprocessing.html#multiprocessing.Process)、[Queue 序列化及退出死锁](https://docs.python.org/3.12/library/multiprocessing.html#pipes-and-queues)、[队列关闭与 join\_thread](https://docs.python.org/3.12/library/multiprocessing.html#multiprocessing.Queue)、[spawn 的参数与入口保护](https://docs.python.org/3.12/library/multiprocessing.html#the-spawn-and-forkserver-start-methods)、[共享状态与清理建议](https://docs.python.org/3.12/library/multiprocessing.html#programming-guidelines)；[pickle 的函数定位及支持类型](https://docs.python.org/3.12/library/pickle.html#what-can-be-pickled-and-unpickled)、[pickle 安全警告](https://docs.python.org/3.12/library/pickle.html)、[subprocess.run 的参数、超时与环境](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)。 |